# Open Sustainability Analyst as a library

End-to-end **OSA** without Streamlit: load [ClimRetrieve](https://github.com/tobischimanski/ClimRetrieve)
labels, run the same `DocumentAnalyzer` the app uses, score retrieval against expert chunks, then
check how stable scores and citations are.

The older Colab notebook cloned a private fork with a GitHub token. This package is public — install
from GitHub, no token.

## Evaluation

**Retrieval vs ClimRetrieve** on 10 labelled sustainability reports and assessment questions.

Charts:
1. Precision and recall
2. Overlap of ClimRetrieve-relevant chunks and OSA-selected chunks

**Robustness** — repeated runs and two configs (`top_k=5` vs `top_k=10`):
- How stable are OSA scores, and how much do they move with top-k?
- How stable is the retrieved chunk set, and are k=5 chunks contained in k=10?
- How consistently does OSA cite the same chunks, and are citations a subset of retrieval?

All tables are written to `notebooks/output/` as CSV.

Set `RUN_LIVE_ANALYSIS = True` (requires `OPENAI_API_KEY`) for PDFs + embeddings + LLM.
Without a key the notebook still downloads labels and selects the 10 reports.


## 0. Install


In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "report_analyst").exists():
    sys.path.insert(0, str(REPO_ROOT))
elif (REPO_ROOT.parent / "report_analyst").exists():
    REPO_ROOT = REPO_ROOT.parent
    sys.path.insert(0, str(REPO_ROOT))

if IN_COLAB:
    # Public repo — no GitHub token.
    %pip install -q "git+https://github.com/climateandtech/report-analyst.git@notebooks/library-e2e-climretrieve" openpyxl matplotlib requests python-dotenv
else:
    %pip install -q openpyxl matplotlib requests


## 1. Configuration


In [ ]:
import os
import uuid
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

RUN_LIVE_ANALYSIS = bool(os.getenv("OPENAI_API_KEY"))
EVALUATION_ID = uuid.uuid4().hex
N_REPORTS = 10
MAX_QUESTIONS = 16
N_RUNS = 3
TOP_K_VALUES = [5, 10]
TOP_K_A, TOP_K_B = TOP_K_VALUES
CHUNK_SIZE = 500
CHUNK_SIZES = [CHUNK_SIZE]  # e.g. [300, 500, 800] for an optional size experiment
CHUNK_OVERLAP = 50
K_VALUES = [1, 3, 5, 10]
QUESTION_SET = "climretrieve"

DATA_DIR = Path("notebooks/data") if Path("notebooks").exists() else Path("data")
OUTPUT_DIR = Path("notebooks/output") if Path("notebooks").exists() else Path("output")
PDF_DIR = DATA_DIR / "climretrieve_pdfs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_LIVE_ANALYSIS=", RUN_LIVE_ANALYSIS)
print("output=", OUTPUT_DIR.resolve())


## 2. ClimRetrieve expert labels

Public files from [tobischimanski/ClimRetrieve](https://github.com/tobischimanski/ClimRetrieve).
We keep Core-16 questions and reports that also exist as PDFs.


In [ ]:
import pandas as pd
import requests

from report_analyst.core.benchmark.library_eval import (
    filter_core_questions,
    normalize_climretrieve_columns,
    select_labelled_reports,
)

CLIMRETRIEVE_REPO = "tobischimanski/ClimRetrieve"
LABELS_URL = (
    f"https://raw.githubusercontent.com/{CLIMRETRIEVE_REPO}/main/"
    "Expert-Annotated%20Relevant%20Sources%20Dataset/ClimRetrieve_base.xlsx"
)
REPORTS_API = f"https://api.github.com/repos/{CLIMRETRIEVE_REPO}/contents/Reports"

labels_xlsx = DATA_DIR / "ClimRetrieve_base.xlsx"
if not labels_xlsx.exists():
    response = requests.get(LABELS_URL, timeout=60)
    response.raise_for_status()
    labels_xlsx.write_bytes(response.content)

raw_labels = pd.read_excel(labels_xlsx)
labels = filter_core_questions(normalize_climretrieve_columns(raw_labels))
print("Columns:", list(raw_labels.columns))
print("Core-16 rows:", len(labels), "reports:", labels["document"].nunique())
display(labels.head())

listing = requests.get(REPORTS_API, timeout=60)
listing.raise_for_status()
pdf_files = [item["name"] for item in listing.json() if str(item.get("name", "")).lower().endswith(".pdf")]
print("PDFs:", len(pdf_files))

selected_reports = select_labelled_reports(raw_labels, pdf_files, n=N_REPORTS)
display(selected_reports)
if len(selected_reports) < N_REPORTS:
    print(f"Warning: only {len(selected_reports)} labelled reports matched PDFs")
selected_reports.to_csv(OUTPUT_DIR / "selected_reports.csv", index=False)


## 3. Download the labelled PDFs


In [ ]:
from urllib.parse import quote

def download_pdf(filename: str) -> Path:
    target = PDF_DIR / filename
    if target.exists() and target.stat().st_size > 0:
        return target
    url = f"https://raw.githubusercontent.com/{CLIMRETRIEVE_REPO}/main/Reports/{quote(filename)}"
    response = requests.get(url, timeout=180)
    response.raise_for_status()
    target.write_bytes(response.content)
    return target

pdf_paths = {}
for _, row in selected_reports.iterrows():
    path = download_pdf(row["pdf_filename"])
    pdf_paths[row["document"]] = path
    kb = path.stat().st_size // 1024
    print(f"{row['document']}: {path.name} ({kb} KB)")


## 4. End-to-end analysis (library, no frontend)

`DocumentAnalyzer` is the same object Streamlit uses: question set, chunk parameters,
`retrieve_chunks` (retrieval only) or `process_document` (answers, scores, citations).


In [ ]:
from report_analyst.core.analyzer import DocumentAnalyzer
from report_analyst.core.benchmark.library_eval import match_question
from report_analyst.core.question_loader import get_question_loader

DocumentAnalyzer.reset_instance()
analyzer = DocumentAnalyzer()
analyzer.update_question_set(QUESTION_SET)
analyzer.update_parameters(CHUNK_SIZE, CHUNK_OVERLAP, TOP_K_A)

osa_questions = get_question_loader().get_questions(QUESTION_SET)
clim_questions = sorted(labels["question"].dropna().unique())
rows = []
for qid, payload in osa_questions.items():
    clim_q = match_question(payload["text"], clim_questions)
    if not clim_q:
        continue
    number = int(str(qid).rsplit("_", 1)[-1])
    rows.append(
        {
            "osa_question_id": qid,
            "osa_question_number": number,
            "osa_text": payload["text"],
            "climretrieve_question": clim_q,
        }
    )
question_map_df = pd.DataFrame(rows).drop_duplicates("climretrieve_question").head(MAX_QUESTIONS)
display(question_map_df)
question_map_df.to_csv(OUTPUT_DIR / "question_map.csv", index=False)
print("Mapped questions:", len(question_map_df))
print("LLM configured:", analyzer.llm is not None)


In [ ]:
async def collect_analysis(file_path, question_numbers, force_recompute=True, use_llm_scoring=False):
    collected = []
    async for item in analyzer.process_document(
        file_path=str(file_path),
        selected_questions=list(question_numbers),
        use_llm_scoring=use_llm_scoring,
        force_recompute=force_recompute,
    ):
        if "error" in item:
            print(item["error"])
            continue
        if "result" in item:
            collected.append(item)
    return collected


if RUN_LIVE_ANALYSIS and question_map_df.empty:
    raise RuntimeError("No OSA questions mapped onto ClimRetrieve Core-16 questions.")

if RUN_LIVE_ANALYSIS:
    demo_doc = selected_reports.iloc[0]["document"]
    demo_pdf = pdf_paths[demo_doc]
    demo_q = question_map_df.iloc[0]
    print("Demo:", demo_doc, demo_q["osa_question_id"])
    retrieved = await analyzer.retrieve_chunks(str(demo_pdf), demo_q["osa_text"], top_k=TOP_K_A)
    print(f"Retrieved {len(retrieved)} chunks")
    for i, chunk in enumerate(retrieved[:3], start=1):
        score = chunk.get("score", chunk.get("similarity_score", 0.0))
        print(f"\n[{i}] score={score:.3f}\n{chunk['text'][:400]}")
    analysis = await collect_analysis(demo_pdf, [int(demo_q["osa_question_number"])])
    if analysis:
        result = analysis[0]["result"]
        print("SCORE:", result.get("SCORE"))
        print("SOURCES:", result.get("SOURCES"))
        print("ANSWER:", str(result.get("ANSWER", ""))[:500])
else:
    print("Skipping live analysis (set OPENAI_API_KEY and RUN_LIVE_ANALYSIS=True).")


## 5. Retrieval quality vs ClimRetrieve

OSA chunks are aligned to expert spans with token overlap, then scored with
`EvaluationEngine.compare_flexible_datasets`.


In [ ]:
from report_analyst.core.benchmark.evaluation_engine import EvaluationEngine
from report_analyst.core.benchmark.library_eval import (
    build_climretrieve_answer_rows,
    build_ground_truth_rows,
    build_osa_retrieval_rows,
    build_overlap_table,
    metrics_to_frame,
)
from report_analyst.core.benchmark.retrieval_results_loader import load_flexible_dataset_from_csv

gt = build_ground_truth_rows(raw_labels, selected_reports["document"].tolist())
gt = gt[gt["question"].isin(set(question_map_df["climretrieve_question"]))]
gt.to_csv(OUTPUT_DIR / "climretrieve_ground_truth.csv", index=False)
expert_answers = build_climretrieve_answer_rows(
    raw_labels,
    selected_reports["document"].tolist(),
    question_map_df["climretrieve_question"].tolist(),
)
expert_answer_lookup = {
    (row.document, row.question): row for row in expert_answers.itertuples(index=False)
}
expert_answers.to_csv(OUTPUT_DIR / "climretrieve_answers.csv", index=False)
print("Ground-truth rows:", len(gt), "queries:", gt["query_id"].nunique())
print("Explicit expert yes/no answers:", expert_answers["expert_yes_no"].notna().sum())

retrieved_by_query = {}
if RUN_LIVE_ANALYSIS:
    analyzer.update_parameters(CHUNK_SIZE, CHUNK_OVERLAP, TOP_K_A)
    for _, report in selected_reports.iterrows():
        pdf = pdf_paths[report["document"]]
        for _, qrow in question_map_df.iterrows():
            chunks = await analyzer.retrieve_chunks(str(pdf), qrow["osa_text"], top_k=TOP_K_A)
            retrieved_by_query[(report["document"], qrow["climretrieve_question"])] = chunks

osa_retrieval = build_osa_retrieval_rows(retrieved_by_query, gt) if retrieved_by_query else pd.DataFrame()
metrics_df = pd.DataFrame()
overlap = pd.DataFrame()
if not osa_retrieval.empty:
    osa_retrieval.to_csv(OUTPUT_DIR / "osa_retrieval.csv", index=False)
    overlap = build_overlap_table(gt, osa_retrieval)
    overlap.to_csv(OUTPUT_DIR / "chunk_overlap.csv", index=False)
    gt_ds = load_flexible_dataset_from_csv(
        csv_path=str(OUTPUT_DIR / "climretrieve_ground_truth.csv"),
        dataset_name="climretrieve",
    )
    osa_ds = load_flexible_dataset_from_csv(
        csv_path=str(OUTPUT_DIR / "osa_retrieval.csv"),
        dataset_name="osa",
    )
    metrics = EvaluationEngine().compare_flexible_datasets(gt_ds, osa_ds, k_values=K_VALUES)
    metrics_df = metrics_to_frame(metrics, config_id=f"top_k_{TOP_K_A}")
    metrics_df.to_csv(OUTPUT_DIR / "retrieval_metrics.csv", index=False)
    display(metrics_df.pivot(index="metric", columns="k", values="value").round(4))
    display(overlap.head())
else:
    print("No retrieval rows. Set RUN_LIVE_ANALYSIS=True to compute metrics.")


### Charts: precision / recall and chunk overlap


In [ ]:
import matplotlib.pyplot as plt

if metrics_df.empty or overlap.empty:
    print("Charts skipped until retrieval CSVs exist.")
else:
    pr = metrics_df[metrics_df["metric"].isin(["precision", "recall"])].dropna(subset=["k"])
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for metric, ax in zip(["precision", "recall"], axes, strict=True):
        subset = pr[pr["metric"] == metric]
        ax.plot(subset["k"], subset["value"], marker="o")
        ax.set_xlabel("k")
        ax.set_ylabel(metric)
        ax.set_title(f"{metric.title()}@k vs ClimRetrieve")
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "precision_recall.png", dpi=120)
    plt.show()

    totals = overlap[["n_climretrieve_only", "n_both", "n_osa_only"]].sum()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(
        ["ClimRetrieve only", "Both", "OSA only"],
        totals.values,
        color=["#6c8ead", "#2a9d8f", "#e9c46a"],
    )
    ax.set_ylabel("Chunks (sum over queries)")
    ax.set_title("Overlap of ClimRetrieve-relevant and OSA-retrieved chunks")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "chunk_overlap.png", dpi=120)
    plt.show()


## 6. Robustness: repeated runs and two top-k configs

Same reports and questions, `N_RUNS` times, at `top_k=5` and `top_k=10`.


In [ ]:
from report_analyst.core.benchmark.library_eval import (
    build_analysis_run_rows,
    citation_consistency,
    citation_subset_rate,
    combine_analysis_run_rows,
    pairwise_chunk_selection,
    retrieved_chunk_consistency,
    score_distribution_summary,
    score_stability,
    topk_retrieved_containment,
    topk_score_delta,
    yes_no_answer_comparison,
)

robustness_rows = []
chunk_score_rows = []
all_result_rows = []
if RUN_LIVE_ANALYSIS:
    analyzer.use_cache = False
    configs = [
        (f"cs{chunk_size}_k{top_k}", chunk_size, top_k)
        for chunk_size in CHUNK_SIZES
        for top_k in TOP_K_VALUES
    ]
    for _, report in selected_reports.iterrows():
        pdf = pdf_paths[report["document"]]
        gt_doc = gt[gt["document"] == report["document"]]
        for config_id, chunk_size, top_k in configs:
            analyzer.update_parameters(chunk_size, CHUNK_OVERLAP, top_k)
            for run_id in range(1, N_RUNS + 1):
                for _, qrow in question_map_df.iterrows():
                    items = await collect_analysis(
                        pdf,
                        [int(qrow["osa_question_number"])],
                        force_recompute=True,
                        use_llm_scoring=True,
                    )
                    result = items[0]["result"] if items else {}
                    gt_q = gt_doc[gt_doc["question"] == qrow["climretrieve_question"]]
                    expert = expert_answer_lookup.get((report["document"], qrow["climretrieve_question"]))
                    context = {
                        "evaluation_id": EVALUATION_ID,
                        "document": report["document"],
                        "pdf_filename": report["pdf_filename"],
                        "question": qrow["climretrieve_question"],
                        "osa_question_id": qrow["osa_question_id"],
                        "config_id": config_id,
                        "top_k": top_k,
                        "chunk_size": chunk_size,
                        "chunk_overlap": CHUNK_OVERLAP,
                        "model": analyzer.default_model,
                        "run_id": run_id,
                        "expert_answer": getattr(expert, "expert_answer", None),
                        "expert_yes_no": getattr(expert, "expert_yes_no", None),
                    }
                    answer_row, chunk_rows = build_analysis_run_rows(result, gt_q, context)
                    robustness_rows.append(answer_row)
                    chunk_score_rows.extend(chunk_rows)
                    all_result_rows.extend(combine_analysis_run_rows(answer_row, chunk_rows))

robustness = pd.DataFrame(robustness_rows)
chunk_scores = pd.DataFrame(chunk_score_rows)
all_results = pd.DataFrame(all_result_rows)
stability = pd.DataFrame()
citations = pd.DataFrame()
retrieved_sets = pd.DataFrame()
selection_pairs = pd.DataFrame()
containment = pd.DataFrame()
deltas = pd.DataFrame()
answer_ranges = pd.DataFrame()
chunk_score_ranges = pd.DataFrame()
selection_ranges = pd.DataFrame()
yes_no_detail = pd.DataFrame()
yes_no_metrics = pd.DataFrame()
if robustness.empty:
    print("Robustness skipped.")
else:
    stability = score_stability(robustness)
    citations = citation_consistency(robustness)
    retrieved_sets = retrieved_chunk_consistency(robustness)
    selection_pairs = pairwise_chunk_selection(robustness)
    low_config = f"cs{CHUNK_SIZES[0]}_k{TOP_K_VALUES[0]}"
    high_config = f"cs{CHUNK_SIZES[0]}_k{TOP_K_VALUES[1]}"
    containment = topk_retrieved_containment(robustness, low_config, high_config)
    deltas = topk_score_delta(stability, low_config, high_config)
    distribution_groups = ("chunk_size", "top_k", "config_id")
    answer_ranges = score_distribution_summary(robustness, "answer_score", distribution_groups)
    chunk_score_ranges = score_distribution_summary(
        chunk_scores, "llm_score", (*distribution_groups, "is_evidence")
    )
    selection_ranges = score_distribution_summary(selection_pairs, "selection_jaccard", distribution_groups)
    yes_no_detail, yes_no_metrics = yes_no_answer_comparison(robustness)
    print(f"Citations contained in retrieval: {citation_subset_rate(robustness):.1%}")
    display(answer_ranges)
    display(chunk_score_ranges)
    display(selection_ranges)
    display(yes_no_metrics)


In [ ]:
if robustness.empty:
    print("Robustness charts skipped.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    plot_specs = [
        (robustness, "answer_score", "Answer score", axes[0]),
        (chunk_scores, "llm_score", "Selected-chunk LLM score", axes[1]),
        (selection_pairs, "selection_jaccard", "Selected-chunk consistency", axes[2]),
    ]
    for frame, value, title, ax in plot_specs:
        groups = [(name, group[value].dropna()) for name, group in frame.groupby("config_id")]
        groups = [(name, values) for name, values in groups if not values.empty]
        if groups:
            ax.boxplot([values for _, values in groups], tick_labels=[name for name, _ in groups])
        ax.set_title(title)
        ax.set_xlabel("Configuration")
        ax.set_ylabel(value)
        ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "robustness_boxplots.png", dpi=150)
    plt.show()


## 7. CSV exports


In [ ]:
from report_analyst.core.benchmark.library_eval import write_eval_csvs

frames = {
    "selected_reports": selected_reports,
    "question_map": question_map_df,
    "climretrieve_ground_truth": gt,
    "climretrieve_answers": expert_answers,
    "climretrieve_labels_core16": labels,
}
if not osa_retrieval.empty:
    frames["osa_retrieval"] = osa_retrieval
    frames["chunk_overlap"] = overlap
    frames["retrieval_metrics"] = metrics_df
if not robustness.empty:
    frames["all_results"] = all_results
    frames["analysis_runs"] = robustness
    frames["chunk_scores"] = chunk_scores
    frames["answer_score_stability"] = stability
    frames["answer_score_ranges"] = answer_ranges
    frames["chunk_llm_score_ranges"] = chunk_score_ranges
    frames["chunk_selection_pairs"] = selection_pairs
    frames["chunk_selection_ranges"] = selection_ranges
    frames["citation_consistency"] = citations
    frames["retrieved_chunk_consistency"] = retrieved_sets
    frames["topk_retrieved_containment"] = containment
    frames["topk_answer_score_delta"] = deltas
    frames["yes_no_answer_comparison"] = yes_no_detail
    frames["yes_no_answer_metrics"] = yes_no_metrics

written = write_eval_csvs(frames, OUTPUT_DIR)
for name, path in written.items():
    print(f"{name}: {path} ({path.stat().st_size} bytes)")
